<a href="https://colab.research.google.com/github/MarcinPlodzien/deep-learning-for-classical-and-quantum-physics/blob/main/notebooks/lecture-08/exercise-04-schroedinger-pinn.ipynb"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open in Colab"></a>

<hr/>
# Deep Learning in Classical and Quantum Physics
2026/2027 <br>
**Author:** Marcin Płodzień <br>
**Exercise:** 04 <br>
**Related labs:** 045
<hr/>

# Exercise 04 — Schrödinger equation with PINNs

A physics-informed neural network (PINN) for the Schrödinger equation $\mathrm i\,\partial_t\psi=\hat H\psi$, $\hat H=-\tfrac12\partial_x^2+V(x)$, in the units $\hbar=m_0=\omega=1$ of the harmonic trap, represents the wave function as $\psi_\theta=u_\theta+\mathrm iv_\theta$. A multilayer perceptron with four tanh layers of 50 units receives the rescaled inputs $x/L$ and $2t/t_{\max}-1$, and its two outputs are multiplied by the envelope $e^{-x^2/4}$. The loss is the mean squared misfit to noisy snapshots of $\psi$ at the times $t_k$, plus the mean squared residuals
$$
f_u=-\partial_tv_\theta+\tfrac12\partial_x^2u_\theta-V\,u_\theta,\qquad
f_v=\partial_tu_\theta+\tfrac12\partial_x^2v_\theta-V\,v_\theta
$$
at $N_r$ collocation points drawn uniformly from $[-L,L]\times[0,t_{\max}]$, plus a term that keeps $\psi_\theta(\pm L,t)$ small. The box has $L=6$ and 256 grid points, a snapshot holds $\psi$ at every second grid point with independent Gaussian noise of standard deviation $\sigma=0.01$ on its real and imaginary parts, and training uses 6000 Adam steps followed by 2000 L-BFGS iterations. The first code cell defines these tools together with the split-step Fourier solver, the energy $E(t)=\int[\tfrac12|\partial_x\psi|^2+V|\psi|^2]dx/N(t)$ with the norm $N(t)=\int|\psi|^2dx$, and the moments of the density.

We work through four exercises. A coherent state tests whether the PINN reproduces a classical trajectory with a packet of constant width. A Crank–Nicolson solver is a second classical reference next to the split-step method. An anharmonic trap removes the exact revivals of the harmonic one and tests the PINN where the motion has no single period. A scan of the number of collocation points shows how densely the residual must be sampled, and a check of the norm drift against the size of the residual closes the notebook. Training runs take about one minute each on a multi-core CPU.

> **How to run this notebook in Google Colab**
> 1. Go to <https://colab.research.google.com> and upload the notebook with *File → Upload notebook*. The first code cell sets `JAX_PLATFORMS=cpu`, and the whole notebook runs in about ten minutes on a multi-core CPU.
> 2. `jax`, `optax`, `scipy` and `matplotlib` are pre-installed in Colab.
> 3. Run the cells in order. Later cells use variables defined earlier.

In [ ]:
import os
os.environ["JAX_PLATFORMS"] = "cpu"     # run JAX on the CPU (set before importing jax)

import json, time
import numpy as np
import matplotlib as mpl
import matplotlib.pyplot as plt
from math import factorial
from scipy.linalg import solve_banded

import jax, jax.numpy as jnp
jax.config.update("jax_enable_x64", True)   # reference solutions in float64, networks in float32
import optax

mpl.rcParams.update({"font.size": 14, "axes.labelsize": 15, "axes.titlesize": 15,
                     "xtick.labelsize": 13, "ytick.labelsize": 13, "legend.fontsize": 12,
                     "figure.titlesize": 16, "lines.linewidth": 2.0,
                     "axes.spines.top": False, "axes.spines.right": False, "legend.frameon": False})
UJBlue, UJNavy, DSOrange, UJGray = "#005CA7", "#002B5C", "#D35A05", "#5A6270"
PURPLE = "#7B4FA0"
OUT = "figures"
os.makedirs(OUT, exist_ok=True)
def save(fig, name):
    for ext in ("pdf", "png"):
        fig.savefig(f"{OUT}/{name}.{ext}", dpi=300, bbox_inches="tight", pad_inches=0.02)
nums = {}
t_start = time.time()

# ---------------------------------------------------------------- grid, eigenstates and reference solvers
L_BOX, N_X = 6.0, 256                          # box [-L, L) with 256 points, as in the trap of the lecture
X = -L_BOX + 2 * L_BOX * np.arange(N_X) / N_X
DX = X[1] - X[0]
K = 2 * np.pi * np.fft.fftfreq(N_X, d=DX)       # wave numbers of the periodic grid

def hermite_functions(n_max, x):
    """phi_0 ... phi_n_max of the harmonic trap by the three-term recurrence."""
    phi = [np.pi**-0.25 * np.exp(-x**2 / 2), np.sqrt(2) * x * np.pi**-0.25 * np.exp(-x**2 / 2)]
    for n in range(1, n_max):
        phi.append(np.sqrt(2 / (n + 1)) * x * phi[n] - np.sqrt(n / (n + 1)) * phi[n - 1])
    return np.array(phi[:n_max + 1])

def split_step(psi0, dt, n_steps, V):
    """Strang splitting exp(-i V dt/2) F^-1 exp(-i k^2 dt/2) F exp(-i V dt/2); returns all n_steps + 1 states."""
    half_v = jnp.exp(-0.5j * dt * jnp.asarray(V)); kin = jnp.exp(-0.5j * dt * jnp.asarray(K)**2)
    def step(psi, _):
        psi = half_v * jnp.fft.ifft(kin * jnp.fft.fft(half_v * psi))
        return psi, psi
    _, psis = jax.lax.scan(step, jnp.asarray(psi0, jnp.complex128), None, length=n_steps)
    return np.asarray(jnp.concatenate([jnp.asarray(psi0, jnp.complex128)[None], psis]))

def fourier_hamiltonian(V):
    """Matrix of -1/2 d^2/dx^2 + V on the periodic grid, kinetic part applied through the discrete Fourier transform."""
    T = np.fft.ifft(0.5 * K[:, None]**2 * np.fft.fft(np.eye(N_X), axis=0), axis=0)
    H = T + np.diag(V)
    return 0.5 * (H + H.conj().T)

def energy(psi, V):
    """E(t) = int [|d_x psi|^2 / 2 + V |psi|^2] dx / N(t), derivative through the discrete Fourier transform."""
    dpsi = np.fft.ifft(1j * K * np.fft.fft(psi, axis=-1), axis=-1)
    dens = np.abs(psi)**2
    return (0.5 * np.abs(dpsi)**2 + V * dens).sum(-1) / dens.sum(-1)

def moments(psi):
    """Norm N(t), mean position <x>(t) and width sigma_x(t) on the grid."""
    dens = np.abs(psi)**2
    norm = dens.sum(-1) * DX
    xm = (dens * X).sum(-1) * DX / norm
    x2 = (dens * X**2).sum(-1) * DX / norm
    return norm, xm, np.sqrt(x2 - xm**2)

# ---------------------------------------------------------------- the PINN of the lecture, in a general form
F32 = jnp.float32
WIDTH, DEPTH = 50, 4
SIZES = [2] + [WIDTH] * DEPTH + [2]
I_DATA = np.arange(0, N_X, 2)                   # every second grid point, 128 positions per snapshot
SIGMA = 0.01                                    # noise on Re psi and Im psi of every snapshot value
N_B = 100                                       # boundary times

def init_mlp(key, sizes=SIZES):
    params = []
    for n_in, n_out in zip(sizes[:-1], sizes[1:]):
        key, sub = jax.random.split(key)
        params.append((jax.random.normal(sub, (n_out, n_in), F32) / float(np.sqrt(n_in)), jnp.zeros(n_out, F32)))
    return params

def make_net(t_max, env_width=2.0):
    """MLP on (x/L, 2t/t_max - 1) with four tanh layers, two outputs multiplied by exp(-x^2/env_width^2)."""
    L_SC, T_SC = jnp.asarray(L_BOX, F32), jnp.asarray(t_max, F32)
    def net(params, x, t):
        h = jnp.stack([x / L_SC, 2 * t / T_SC - 1], -1)
        for W, b in params[:-1]:
            h = jnp.tanh(h @ W.T + b)
        W, b = params[-1]
        return (h @ W.T + b) * jnp.exp(-x**2 / env_width**2)[:, None]
    return net

def residual(net, params, x, t, V):
    """Real and imaginary parts f_u, f_v of i psi_t + psi_xx/2 - V psi, derivatives by forward-mode autodiff."""
    f = lambda x_, t_: net(params, x_, t_)
    uv, uv_t = jax.jvp(lambda t_: f(x, t_), (t,), (jnp.ones_like(t),))
    d_x = lambda x_: jax.jvp(lambda y: f(y, t), (x_,), (jnp.ones_like(x_),))[1]
    uv_xx = jax.jvp(d_x, (x,), (jnp.ones_like(x),))[1]
    f_u = -uv_t[:, 1] + 0.5 * uv_xx[:, 0] - V * uv[:, 0]
    f_v = uv_t[:, 0] + 0.5 * uv_xx[:, 1] - V * uv[:, 1]
    return f_u, f_v

def make_snapshots(psi_grid_at, t_snap, seed=0):
    """Noisy snapshots at every second grid point; psi_grid_at(t) returns psi on the full grid."""
    rng = np.random.default_rng(seed)
    psi = np.stack([psi_grid_at(tk)[I_DATA] for tk in t_snap])
    psi = psi + SIGMA * (rng.normal(size=psi.shape) + 1j * rng.normal(size=psi.shape))
    XD, TD = np.meshgrid(X[I_DATA], t_snap)
    return (jnp.asarray(XD.ravel(), F32), jnp.asarray(TD.ravel(), F32),
            jnp.asarray(np.stack([psi.real.ravel(), psi.imag.ravel()], 1), F32))

def collocation(n_col, t_max, seed=1):
    key = jax.random.key(seed)
    return (jax.random.uniform(key, (n_col,), F32, -L_BOX, L_BOX),
            jax.random.uniform(jax.random.fold_in(key, 1), (n_col,), F32, 0.0, t_max))

def make_loss(net, data, col, V_fn, t_max, w_r=1.0):
    """Snapshot misfit + w_r * mean squared residual at the collocation points + boundary term."""
    xd, td, uvd = data
    xc, tc = col
    Vc = V_fn(xc)
    t_b = jnp.linspace(0.0, t_max, N_B, dtype=F32)
    def loss(params):
        L = jnp.mean(jnp.sum((net(params, xd, td) - uvd)**2, 1))
        L += jnp.mean(jnp.sum(net(params, jnp.full(N_B, L_BOX, F32), t_b)**2, 1)
                      + jnp.sum(net(params, jnp.full(N_B, -L_BOX, F32), t_b)**2, 1)) / 2
        if w_r > 0:
            f_u, f_v = residual(net, params, xc, tc, Vc)
            L += w_r * jnp.mean(f_u**2 + f_v**2)
        return L
    return loss

def train(loss, params, n_adam=6000, n_lbfgs=2000, lr=2e-3):
    """Adam with a cosine learning-rate decay, followed by L-BFGS, as in the lecture."""
    tx = optax.adam(optax.cosine_decay_schedule(lr, n_adam, alpha=1e-2))
    state = tx.init(params)
    @jax.jit
    def adam_step(p, s):
        l, g = jax.value_and_grad(loss)(p)
        u, s = tx.update(g, s, p)
        u = jax.tree.map(lambda a, b: a.astype(b.dtype), u, p)
        return optax.apply_updates(p, u), s, l
    lb = optax.lbfgs()
    vg_state = optax.value_and_grad_from_state(loss)
    @jax.jit
    def lbfgs_step(p, s):
        l, g = vg_state(p, state=s)
        u, s = lb.update(g, s, p, value=l, grad=g, value_fn=loss)
        u = jax.tree.map(lambda a, b: a.astype(b.dtype), u, p)
        return optax.apply_updates(p, u), s, l
    t0 = time.time()
    for it in range(n_adam + n_lbfgs):
        if it == n_adam:
            state = lb.init(params)
        params, state, l = (adam_step if it < n_adam else lbfgs_step)(params, state)
    return params, dict(time=time.time() - t0, final_loss=float(loss(params)))

def predict(net, params, t_grid):
    """psi_theta on the full grid at the times t_grid, shape (len(t_grid), N_X), complex128."""
    xg = jnp.asarray(np.broadcast_to(X, (len(t_grid), N_X)).ravel(), F32)
    tg = jnp.asarray(np.broadcast_to(np.asarray(t_grid)[:, None], (len(t_grid), N_X)).ravel(), F32)
    uv = np.asarray(net(params, xg, tg), np.float64).reshape(len(t_grid), N_X, 2)
    return uv[..., 0] + 1j * uv[..., 1]
print("jax", jax.__version__, "| optax", optax.__version__)

<hr/>
## 1. A coherent state

### Exercise 1.1. Classical motion of a displaced ground state

Prepare the ground state of the harmonic trap displaced to $x_0=1.5$, $\psi(x,0)=\pi^{-1/4}e^{-(x-x_0)^2/2}$. Show that its expansion coefficients in the eigenstates $\phi_n$ have the Poisson weights $|c_n|^2=e^{-\alpha^2}\alpha^{2n}/n!$ with $\alpha=x_0/\sqrt2$, and that its energy is $\langle\hat H\rangle=\tfrac12+\tfrac12x_0^2$. Show that the density moves as $|\psi(x,t)|^2=\phi_0^2(x-x_0\cos t)$, so that $\langle x\rangle(t)=x_0\cos t$ and the width $\sigma_x(t)=(\langle x^2\rangle-\langle x\rangle^2)^{1/2}$ stays at $1/\sqrt2$. Check all statements with the split-step solver over one period with 1024 steps. Then train the PINN on three snapshots at $t=0$, 0.5 and 1 over $0\le t\le2\pi$ with 3072 collocation points, and compare its $\langle x\rangle(t)$, $\sigma_x(t)$, norm and energy with the exact values.

In [ ]:
# your code: Poisson weights and energy; split-step over one period; PINN on three snapshots; <x>(t) and sigma_x(t)

<hr/>
## 2. Crank–Nicolson and split-step references

### Exercise 2.1. A second classical solver

The Crank–Nicolson method discretises the kinetic energy with the three-point second difference on the grid, with zero values outside the box, and advances the state by
$$
\Big(1+\tfrac{\mathrm i}{2}\Delta t\,H\Big)\psi_{n+1}=\Big(1-\tfrac{\mathrm i}{2}\Delta t\,H\Big)\psi_n ,
$$
where $H$ is the tridiagonal matrix of the discretised Hamiltonian. Show that the update matrix $(1+\tfrac{\mathrm i}2\Delta tH)^{-1}(1-\tfrac{\mathrm i}2\Delta tH)$ is unitary for Hermitian $H$ and that it conserves the discrete energy $\psi^\dagger H\psi$ exactly. Implement the method with `scipy.linalg.solve_banded`, propagate $(\phi_0+\phi_1)/\sqrt2$ of the harmonic trap to $t=2\pi$ with 64 to 4096 steps on grids of 256, 512 and 1024 points, and record the largest error against the exact solution, the norm and the spread of $\psi^\dagger H\psi\,\Delta x$. Explain the error floor with the eigenvalues of the three-point Hamiltonian. Compare with the split-step solver on 256 points, including the spread of its energy $E(t)$ over the run.

In [ ]:
# your code: Crank-Nicolson with solve_banded on 256, 512, 1024 points; eigenvalues of the three-point Hamiltonian;
# split-step on 256 points; error against the step

<hr/>
## 3. An anharmonic trap

### Exercise 3.1. Spectrum, level spacings and revivals

Add a quartic term to the trap, $V(x)=\tfrac12x^2+\lambda x^4$ with $\lambda=0.1$, and prepare the state $\psi(x,0)=(\phi_0+\phi_1+\phi_2)/\sqrt3$ built from the eigenstates of the harmonic trap. Compute the lowest eigenvalues $E_n$ and eigenstates by diagonalising the Hamiltonian on the Fourier grid, expand the initial state in them, and check the eigenstate expansion against the split-step solver at $t=2\pi$. In the harmonic trap ($\lambda=0$) the three components have the spacings 1, 1 and 2, and the density returns to its initial form after $2\pi$. Explain from the level spacings why the anharmonic trap has no exact revival, compute the return probability $F(t)=|\langle\psi(0)|\psi(t)\rangle|^2$ for $0\le t\le300$, and find its largest values after $t=1$ in the windows up to $t=10$, 50, 100 and 300.

In [ ]:
# your code: Fourier-grid Hamiltonian with the quartic term; expansion of the initial state; F(t) and its maxima

### Exercise 3.2. PINN in the anharmonic trap

Train the PINN with the potential $V(x)=\tfrac12x^2+0.1\,x^4$ in its residual on three snapshots at $t=0$, 0.5 and 1 of the anharmonic evolution, once over $0\le t\le2\pi$ with 3072 collocation points and once over $0\le t\le4\pi$ with 6144 points, with the training budget of the lecture. Report the largest error inside and after the snapshot window, the norm and the energy $E(t)$ of the PINN, and compare the energy with the exact $\langle\hat H\rangle$. Decide from the energy alone whether the PINN solution is trustworthy.

In [ ]:
# your code: PINN with the anharmonic potential on [0, 2 pi] and [0, 4 pi]; error, norm and energy

<hr/>
## 4. Collocation density and the norm

### Exercise 4.1. How many collocation points

For the superposition $(\phi_0+\phi_1)/\sqrt2$ of the harmonic trap and the three snapshots at $t=0$, 0.5 and 1, train the PINN with $N_r=96$, 192, 384, 768, 1536, 3072 and 6144 collocation points over $0\le t\le2\pi$ with half the training budget, 3000 Adam steps and 1000 L-BFGS iterations, from the same initial weights. For each run report the mean squared residual at the collocation points and at 20000 fresh points, the largest error after the snapshots, and the largest deviations of the norm and of the energy from 1. Explain the difference between the two residuals for small $N_r$.

In [ ]:
# your code: collocation scan with half the budget; residual at training and fresh points; error, norm, energy

### Exercise 4.2. Norm drift and the size of the residual

For a wave function that vanishes at the edges of the box the norm of the network changes as $dN_\theta/dt=2\,\mathrm{Im}\langle\psi_\theta|r_\theta\rangle$ with the complex residual $r_\theta=\mathrm i\partial_t\psi_\theta-\hat H\psi_\theta$, and the Cauchy–Schwarz inequality gives $|dN_\theta/dt|\le2\|\psi_\theta\|\,\|r_\theta\|$. For the run with $N_r=3072$ of Exercise 4.1 compute $dN_\theta/dt=2\int(u_\theta\partial_tu_\theta+v_\theta\partial_tv_\theta)dx$ with automatic differentiation in $t$, the norm $\|r_\theta(\cdot,t)\|$ of the residual on the grid, and the ratio $|dN_\theta/dt|/(2\|\psi_\theta\|\|r_\theta\|)$ at 60 times, and check the inequality.

In [ ]:
# your code: dN/dt by forward-mode differentiation, residual norm on the grid, ratio at 60 times

<hr/>
### References


- M. Raissi, P. Perdikaris and G. E. Karniadakis, Physics-informed neural networks, J. Comput. Phys. **378**, 686 (2019), [doi:10.1016/j.jcp.2018.10.045](https://doi.org/10.1016/j.jcp.2018.10.045).
- J. Crank and P. Nicolson, A practical method for numerical evaluation of solutions of partial differential equations of the heat-conduction type, Math. Proc. Camb. Philos. Soc. **43**, 50 (1947), [doi:10.1017/S0305004100023197](https://doi.org/10.1017/S0305004100023197).
- J. A. Fleck, J. R. Morris and M. D. Feit, Time-dependent propagation of high energy laser beams through the atmosphere, Appl. Phys. **10**, 129 (1976), [doi:10.1007/BF00896333](https://doi.org/10.1007/BF00896333).
- G. Strang, On the construction and comparison of difference schemes, SIAM J. Numer. Anal. **5**, 506 (1968), [doi:10.1137/0705041](https://doi.org/10.1137/0705041).
- D. J. Griffiths and D. F. Schroeter, *Introduction to Quantum Mechanics*, 3rd ed., Cambridge University Press (2018), [doi:10.1017/9781316995433](https://doi.org/10.1017/9781316995433).